# NB02 — fPINN de identificación del TCLab (modo simulación)

**Proyecto:** fPINN + sintonía diferenciable de FOPID en el TCLab — *Control Engineering Practice*.
**Sección del manuscrito:** 4.2 *Identification* (tabla `tab:sim_id`) y figura de RMSE/|β̂−β*| contra la duración de los datos.

**Qué hace.** Genera datos con una **planta virtual fraccionaria** (balance de energía 2×2 del TCLab con D^α en el
calentador y D^β en el sensor), entrena la **fPINN** (red de trayectorias tipo Pang + residuo de Grünwald–Letnikov,
Adam → L-BFGS) y la **PINN entera** (α = β = 1, para B3), y compara contra los modelos de referencia ajustados con los
mismos datos: modelo físico entero y fraccionario por mínimos cuadrados de error de salida (como NB01), FOPDT, SOPDT y FFOPDT.

| Factor | Niveles |
|---|---|
| Caso (α*, β*) | (1, 1) control · (1, 0,7) sensor fraccionario · (0,9, 0,7) calentador y sensor |
| Ruido σn | 0 · 0,1 · 0,3 °C, siempre + cuantización 0,322 °C (ADC 10 bits) y PWM de 8 bits |
| Datos de entrenamiento | primeros 15 · 30 · 45 min de **cada** experimento A, B y C |
| Semillas | 5 (ruido + inicialización de la red) |

**Parámetros identificados:** ϑ = (U, α1, α2, τs, α, β); m, cp, A, As, ε fijos (tabla `tab:tclab_par`).
**El sustituto para NB03 es el modelo fraccionario identificado** (la red se descarta): se exporta ϑ̂ en JSON.

**Validación.** Dos experimentos virtuales independientes, nunca usados para entrenar, y comparados con la temperatura
**sin ruido** de la planta: **V1** = APRBS en Q1 (semilla 17, Q2 = 0, 45 min) → RMSE de T1, la métrica de la tabla
(todos los modelos, también los SISO); **V2** = APRBS en Q1 y Q2 a la vez (semillas 18/19) → RMSE de T1 y T2 (solo modelos 2×2).

**Cómo usarlo.** `MODE = 'quick'` corre una configuración mínima (prueba de que todo funciona, ~2–4 min).
`MODE = 'full'` corre el barrido completo (135 corridas de fPINN + 15 de PINN entera, del orden de 20 h en una CPU
de 4 núcleos; **empieza por la configuración principal** σn = 0,1 °C y 30 min, de modo que la tabla y los sustitutos
para NB03 están listos tras las primeras ~15 corridas, unas 4–5 h). Guarda cada corrida en `NB02_out/resultados_id.csv` en cuanto termina y,
si se interrumpe, **retoma** donde quedó (no repite corridas ya guardadas).
Requiere PyTorch: `%pip install torch` (CPU basta).

In [ ]:
import os
# Windows/Anaconda: evita que el kernel muera por dos copias de OpenMP (torch + numpy). Debe ir antes de importar numpy y torch.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import json, time, math, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)

MODE = 'quick'                 # 'quick' (prueba) | 'full' (barrido completo, varias horas)
OUT_DIR = 'NB02_out'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(os.path.join(OUT_DIR, 'surrogates'), exist_ok=True)
RES_CSV = os.path.join(OUT_DIR, 'resultados_id.csv')
N_THREADS = None               # p. ej. 4 para limitar los hilos de PyTorch; None = valor por defecto
if N_THREADS: torch.set_num_threads(N_THREADS)

CASES = [(1.0, 1.0), (1.0, 0.7), (0.9, 0.7)]
NOISES = [0.0, 0.1, 0.3]
DURATIONS_MIN = [15, 30, 45]
SEEDS = [0, 1, 2, 3, 4]
MAIN = dict(noise=0.1, D=30)   # configuración de la tabla tab:sim_id

# fPINN (ver sección 2). En una CPU de 4 núcleos, una corrida con 15 min de datos tarda ~6 min (Adam ~16 it/s + L-BFGS);
# con 30–45 min, ~8–12 min. Con menos iteraciones β no termina de converger (probado: 2000+500 da β̂≈0,77 para β*=0,7).
PINN = dict(n_adam=4000, n_lbfgs=1500, lr=2e-3, w_r=1.0, w_0=10.0, order_max=1.2,
            width=64, depth=4, n_ff=8, use_z=True, sensor_exact=True)
TH0 = dict(U=8.0, alpha1=0.01, alpha2=0.0075, tau_s=15.0, alpha=0.95, beta=0.95)   # arranque (no la verdad)
L_SUR = None                   # memoria del sustituto en pasos de 1 s (None = completa); ver sección 1
SISO_STARTS = 3
RUN_INT_PINN_ALL = False       # True = PINN entera en todas las configuraciones (por defecto, solo en MAIN)
RUN_SISO_ALL = False           # True = FOPDT/SOPDT/FFOPDT en todas (por defecto, solo en MAIN; el FFOPDT es lento)

if MODE == 'quick':
    CASES, NOISES, DURATIONS_MIN, SEEDS = [(1.0, 0.7)], [0.1], [15], [0]
    MAIN = dict(noise=0.1, D=15)
    PINN.update(n_adam=200, n_lbfgs=20)
    SISO_STARTS = 1
    RES_CSV = os.path.join(OUT_DIR, 'resultados_id_quick.csv')

# Criterio de H1 (el mismo de NB01)
ORDER_TOL, GAIN_TOL = 0.05, 10.0

## 1. Planta virtual fraccionaria y experimentos

Modelo (ec. `eq:fracmodel`), en variables de desviación x = T − T∞:
m·cp·t_c^(α−1)·D^α x_H,i = F_i(x_H, Q_i; U, α1, α2) y τs^β·D^β x_S,i = x_H,i − x_S,i, con F_i el balance de energía
(convección + radiación al ambiente + intercambio con el otro calentador por A_s con el mismo U).
Se integra con GL explícito (ec. `eq:glsim`) con **paso de 0,25 s y memoria completa**: la planta es más fina que el
sustituto (paso 1 s), así que el error de discretización del sustituto no queda oculto.
Experimentos A, B y C idénticos a NB01 (mismos escalones y misma APRBS, semilla 7). El PWM de 8 bits se aplica a la
entrada que recibe la planta; el registro guarda el valor comandado, como con el kit.

In [ ]:
TS = 1.0
STEP_PLAN = [(0, 0), (10, 50), (910, 80), (1810, 20)]
T_END = 2710
C_T_END, C_HOLD, C_LEVELS, C_MIN_JUMP, C_SEED = 3600, (20, 300), (20, 80), 15, 7
V_T_END = 2700
QUANT = 0.322
PHYS = dict(m=4e-3, cp=500.0, A=10e-4, As=2e-4, eps=0.9, sigma=5.67e-8, Ta=23.0, tc=1.0)
TRUE = dict(U=5.0, alpha1=0.01, alpha2=0.0075, tau_s=20.0)
TH_NAMES = ['U', 'alpha1', 'alpha2', 'tau_s', 'alpha', 'beta']

def step_profile(t, plan):
    level = 0.0
    for t0, q in plan:
        if t >= t0: level = q
    return level

def aprbs_plan(t_end=C_T_END, hold=C_HOLD, levels=C_LEVELS, min_jump=C_MIN_JUMP, t0=10, seed=C_SEED):
    """Igual que NB01: niveles y permanencias aleatorios -> [(t_inicio, nivel)]."""
    rng = np.random.default_rng(seed)
    plan, t, last = [(0, 0.0)], t0, 0.0
    while t < t_end:
        q = rng.uniform(*levels)
        while abs(q - last) < min_jump: q = rng.uniform(*levels)
        plan.append((int(t), round(float(q), 1))); last = q
        t += int(round(rng.uniform(*hold)))
    return plan

def make_exp(name):
    """Devuelve t, Q1, Q2 de los experimentos A, B, C (NB01) y de validación V1, V2."""
    prof = lambda plan, t: np.array([step_profile(x, plan) for x in t])
    if name in ('A', 'B'):
        t = np.arange(0, T_END, TS); q = prof(STEP_PLAN, t)
        return (t, q, 0 * q) if name == 'A' else (t, 0 * q, q)
    if name == 'C':
        t = np.arange(0, C_T_END, TS); q = prof(aprbs_plan(), t); return t, q, 0 * q
    t = np.arange(0, V_T_END, TS)
    if name == 'V1':
        q = prof(aprbs_plan(seed=17, t_end=V_T_END), t); return t, q, 0 * q
    if name == 'V2':
        return (t, prof(aprbs_plan(seed=18, t_end=V_T_END), t),
                prof(aprbs_plan(seed=19, t_end=V_T_END, levels=(0, 60), hold=(120, 600)), t))

def pwm8(q):
    return np.round(np.asarray(q, float) * 255 / 100) * 100 / 255

def gl_w(a, n):
    """Pesos de Grünwald-Letnikov w_0..w_{n-1} del orden a."""
    return np.cumprod(np.r_[1.0, 1.0 - (a + 1.0) / np.arange(1, n)])

def F_heat(x1, x2, q1, q2, U, a1, a2, TaK):
    es, A, As = PHYS['eps'] * PHYS['sigma'], PHYS['A'], PHYS['As']
    T1, T2 = TaK + x1, TaK + x2
    ex = U * As * (T2 - T1) + es * As * (T2**4 - T1**4)
    return (U * A * (TaK - T1) + es * A * (TaK**4 - T1**4) + ex + a1 * q1,
            U * A * (TaK - T2) + es * A * (TaK**4 - T2**4) - ex + a2 * q2)

def sim_frac(Q1, Q2, th, Ta, sub=1, L=None):
    """GL explícito (eq:glsim) del modelo 2x2. Entradas ZOH cada 1 s, paso de integración 1/sub s.
    L = memoria en pasos de integración (None = completa). Devuelve T1, T2 medidas y los 4 estados cada 1 s."""
    U, a1, a2, ts_, al, be = (th[k] for k in TH_NAMES)
    mcp = PHYS['m'] * PHYS['cp']; TaK = Ta + 273.15
    dt = TS / sub; N = len(Q1) * sub; Lm = N if L is None else min(int(L), N)
    wa, wb = gl_w(al, Lm + 1), gl_w(be, Lm + 1)
    ca = dt**al / (mcp * PHYS['tc']**(al - 1)); cb = (dt / ts_)**be
    X = np.zeros((N + 1, 4))                  # xH1, xH2, xS1, xS2
    for k in range(1, N + 1):
        j = (k - 1) // sub; xp = X[k - 1]
        f1, f2 = F_heat(xp[0], xp[1], Q1[j], Q2[j], U, a1, a2, TaK)
        i0 = max(0, k - Lm)
        hist = X[k - 1:(i0 - 1 if i0 > 0 else None):-1]; n = hist.shape[0]
        ma = wa[1:n + 1] @ hist[:, :2]; mb = wb[1:n + 1] @ hist[:, 2:]
        X[k] = [ca * f1 - ma[0], ca * f2 - ma[1], cb * (xp[0] - xp[2]) - mb[0], cb * (xp[1] - xp[3]) - mb[1]]
    Xs = X[::sub][:len(Q1)]
    return Ta + Xs[:, 2], Ta + Xs[:, 3], Xs

def measure(T, sigma, rng):
    return np.round((T + rng.normal(0, sigma, T.shape)) / QUANT) * QUANT if sigma > 0 else np.round(T / QUANT) * QUANT

# Trayectorias limpias de la planta: una vez por caso (el ruido se agrega por semilla)
PLANT = {}
for case in CASES:
    th = dict(TRUE, alpha=case[0], beta=case[1]); PLANT[case] = {}
    for nm in ['A', 'B', 'C', 'V1', 'V2']:
        t, q1, q2 = make_exp(nm)
        T1, T2, _ = sim_frac(pwm8(q1), pwm8(q2), th, PHYS['Ta'], sub=4)
        PLANT[case][nm] = dict(t=t, Q1=q1, Q2=q2, T1=T1, T2=T2)
    print('planta', case, 'lista; T1 máx. en C = %.1f °C' % PLANT[case]['C']['T1'].max())

def noisy_data(case, sigma, seed, D_min):
    """Primeros D_min minutos de A, B y C con ruido + cuantización (semilla = seed)."""
    rng = np.random.default_rng(1000 + seed); n = int(D_min * 60 / TS); out = []
    for nm in ['A', 'B', 'C']:
        e = PLANT[case][nm]
        T1, T2 = measure(e['T1'], sigma, rng), measure(e['T2'], sigma, rng)
        out.append(dict(name=nm, t=e['t'][:n], Q1=e['Q1'][:n], Q2=e['Q2'][:n], T1=T1[:n], T2=T2[:n]))
    return out

In [ ]:
# Verificación: error de discretización y de memoria truncada del sustituto (paso 1 s) frente a la planta (0,25 s)
chk = []
for case in CASES:
    th = dict(TRUE, alpha=case[0], beta=case[1]); e = PLANT[case]['C']
    for L in [None, 1800, 900]:
        T1, _, _ = sim_frac(pwm8(e['Q1']), pwm8(e['Q2']), th, PHYS['Ta'], sub=1, L=L)
        chk.append(dict(caso=str(case), memoria_L=('completa' if L is None else L),
                        RMSE_vs_planta=float(np.sqrt(np.mean((T1 - e['T1'])**2)))))
chk = pd.DataFrame(chk); chk.to_csv(os.path.join(OUT_DIR, 'chequeo_memoria.csv'), index=False)
print(chk.pivot(index='caso', columns='memoria_L', values='RMSE_vs_planta').round(3))

case = CASES[-1]
fig, ax = plt.subplots(2, 3, figsize=(14, 5), sharex='col')
for j, nm in enumerate('ABC'):
    e = PLANT[case][nm]; d = noisy_data(case, 0.3, 0, 60)[j]
    ax[0, j].plot(d['t'], d['T1'], '.', ms=1, c='0.5'); ax[0, j].plot(e['t'], e['T1'], label='T1')
    ax[0, j].plot(e['t'], e['T2'], label='T2'); ax[0, j].set_title(f'Exp. {nm}, caso {case}'); ax[0, j].legend()
    ax[1, j].step(e['t'], e['Q1'], label='Q1'); ax[1, j].step(e['t'], e['Q2'], label='Q2'); ax[1, j].set_xlabel('t [s]')
ax[0, 0].set_ylabel('°C'); ax[1, 0].set_ylabel('%'); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig01_planta_virtual.png'), dpi=150); plt.show()

## 2. fPINN

- **Red de trayectorias** (Pang et al., 2019): una red por experimento, 𝒩_w(t) → (x̂_H1, x̂_H2). Entradas: t normalizado,
  8 pares de rasgos de Fourier y la entrada Q filtrada con 4 constantes de tiempo (5, 20, 80, 300 s). Estos últimos son
  funciones conocidas de t; sin ellos la red no logra reproducir los quiebres de x_H en cada cambio de Q y el residuo
  queda 100 veces por encima del piso de discretización (probado en el prototipo).
- **Sensor exacto** (`sensor_exact=True`): como la ecuación del sensor es lineal, x̂_S se obtiene **resolviendo** el GL
  implícito de τs^β D^β x_S + x_S = x̂_H (inversa de serie de potencias por FFT, diferenciable en τs y β). Así β entra
  directamente en la pérdida de datos. Con `False` la red da las 4 salidas y el sensor va como residuo blando (versión del texto actual).
- **Residuo** del calentador con D^α por GL de memoria completa (convolución FFT) sobre la rejilla de 1 s.
- **Pérdida** 𝓛 = 𝓛_data + w_r 𝓛_res + w_0 𝓛_0. Órdenes α, β = 1,2·σ(·) (pueden pasar de 1, para que el caso entero
  no quede pegado a una cota; un orden > 1 se interpreta como orden entero faltante, igual que en NB01).
- **Entrenamiento:** arranque solo con datos (¼ de las iteraciones de Adam), Adam conjunto (w, ϑ) con tasa decreciente, y L-BFGS.
- **PINN entera:** lo mismo con α = β = 1 fijos (baseline B3).

In [ ]:
def gl_w_t(a, n):
    k = torch.arange(1, n, dtype=torch.float64)
    return torch.cumprod(torch.cat([torch.ones(1), 1.0 - (a + 1.0) / k]), 0)

def fft_conv(a, x, N):
    """Convolución causal de la serie a (N) con cada columna de x (N x m), truncada a N."""
    n = 1 << (2 * N - 1).bit_length()
    return torch.fft.irfft(torch.fft.rfft(a, n).unsqueeze(1) * torch.fft.rfft(x, n, dim=0), n, dim=0)[:N]

def gl_deriv(x, a, h=TS):
    return fft_conv(gl_w_t(a, x.shape[0]), x, x.shape[0]) * h ** (-a)

def series_inv(a, N):
    """Recíproco de la serie de potencias a (b * a = delta), N términos, por Newton con productos FFT."""
    b = 1.0 / a[:1]; m = 1
    while m < N:
        m = min(2 * m, N)
        bp = torch.cat([b, torch.zeros(m - b.shape[0])])
        corr = -fft_conv(a[:m], bp.unsqueeze(1), m)[:, 0]; corr[0] = corr[0] + 2.0
        b = fft_conv(bp, corr.unsqueeze(1), m)[:, 0]
    return b

def sensor_solve(xH, ts_, be, h=TS):
    """Solución discreta exacta de tau_s^beta D^beta xS + xS = xH (GL implícito, estado inicial nulo)."""
    N = xH.shape[0]; a = (ts_ / h) ** be * gl_w_t(be, N); a = torch.cat([a[:1] + 1.0, a[1:]])
    return fft_conv(series_inv(a, N), xH, N)

FILTER_TAUS = (5.0, 20.0, 80.0, 300.0)

def input_features(Q1, Q2):
    out = []
    for q in (Q1, Q2):
        for tau in FILTER_TAUS:
            a = math.exp(-TS / tau); z = np.zeros(len(q))
            for k in range(1, len(q)): z[k] = a * z[k - 1] + (1 - a) * q[k - 1]
            out.append(z / 100.0)
    return torch.tensor(np.stack(out, 1))

class TrajNet(nn.Module):
    def __init__(self, t_end, width, depth, n_ff, n_z, n_out, seed, scale=40.0):
        super().__init__()
        g = torch.Generator().manual_seed(seed); self.t_end, self.scale = t_end, scale
        self.register_buffer('freq', math.pi * torch.arange(1, n_ff + 1, dtype=torch.float64))
        layers, d = [], 1 + 2 * n_ff + n_z
        for _ in range(depth): layers += [nn.Linear(d, width), nn.Tanh()]; d = width
        self.net = nn.Sequential(*layers, nn.Linear(d, n_out))
        for m in self.net:
            if isinstance(m, nn.Linear): nn.init.xavier_normal_(m.weight, generator=g); nn.init.zeros_(m.bias)
    def forward(self, t, z=None):
        s = (t / self.t_end).unsqueeze(1)
        f = [s, torch.sin(s * self.freq), torch.cos(s * self.freq)] + ([] if z is None else [z])
        return self.scale * self.net(torch.cat(f, 1))

class Params(nn.Module):
    """theta = (U, alpha1, alpha2, tau_s, alpha, beta): positivos por logaritmo, órdenes = order_max * sigmoid."""
    def __init__(self, th0, learn_orders=True, order_max=1.2):
        super().__init__()
        self.order_max, self.learn_orders = order_max, learn_orders
        self.logp = nn.Parameter(torch.log(torch.tensor([th0[k] for k in TH_NAMES[:4]])))
        logit = lambda o: math.log(o / order_max / (1 - o / order_max))
        if learn_orders: self.o = nn.Parameter(torch.tensor([logit(th0['alpha']), logit(th0['beta'])]))
    def get(self):
        U, a1, a2, ts_ = torch.exp(self.logp)
        if self.learn_orders: al, be = self.order_max * torch.sigmoid(self.o)
        else: al = be = torch.tensor(1.0)
        return U, a1, a2, ts_, al, be
    def as_dict(self):
        return dict(zip(TH_NAMES, [float(v.detach()) for v in self.get()]))

def heater_residual(X, q1, q2, P, TaK, t_ref=100.0):
    """Residuo de m cp t_c^(alpha-1) D^alpha xH = F (forma hacia atrás: entrada del intervalo anterior), escalado a K."""
    U, a1, a2, ts_, al, be = P.get()
    DH = gl_deriv(X[:, :2], al)
    q1p, q2p = torch.cat([q1[:1] * 0, q1[:-1]]), torch.cat([q2[:1] * 0, q2[:-1]])
    f1, f2 = F_heat(X[:, 0], X[:, 1], q1p, q2p, U, a1, a2, TaK)
    c = PHYS['m'] * PHYS['cp'] * PHYS['tc'] ** (al - 1)
    return t_ref * torch.stack([DH[:, 0] - f1 / c, DH[:, 1] - f2 / c], 1)[1:]

def sensor_residual(X, P):
    U, a1, a2, ts_, al, be = P.get()
    return (ts_ ** be * gl_deriv(X[:, 2:], be) - (X[:, :2] - X[:, 2:]))[1:]

def train_pinn(exps, th0=TH0, learn_orders=True, seed=0, n_adam=4000, n_lbfgs=1500, lr=2e-3, w_r=1.0, w_0=10.0,
               order_max=1.2, width=64, depth=4, n_ff=8, use_z=True, sensor_exact=True, verbose=False):
    """Entrena la fPINN (o la PINN entera con learn_orders=False). Devuelve theta_hat, historial y tiempo."""
    torch.manual_seed(seed)
    P = Params(th0, learn_orders, order_max); data = []
    for i, e in enumerate(exps):
        Ta = float(np.mean(np.r_[e['T1'][:5], e['T2'][:5]]))
        data.append(dict(t=torch.tensor(e['t'], dtype=torch.float64), q1=torch.tensor(e['Q1'], dtype=torch.float64),
                         q2=torch.tensor(e['Q2'], dtype=torch.float64), TaK=Ta + 273.15,
                         y=torch.tensor(np.c_[e['T1'], e['T2']]) - Ta,
                         z=input_features(e['Q1'], e['Q2']) if use_z else None,
                         net=TrajNet(float(e['t'][-1]) + 1.0, width, depth, n_ff, 2 * len(FILTER_TAUS) if use_z else 0,
                                     2 if sensor_exact else 4, seed=1000 * seed + i)))
    nets = nn.ModuleList([d['net'] for d in data])

    def states(d):
        X = d['net'](d['t'], d['z'])
        if not sensor_exact: return X
        U, a1, a2, ts_, al, be = P.get()
        return torch.cat([X, sensor_solve(X, ts_, be)], 1)

    def loss_fn(parts=False):
        Ld = Lr = L0 = 0.0
        for d in data:
            X = states(d)
            Ld = Ld + torch.mean((X[:, 2:] - d['y']) ** 2)
            Lr = Lr + torch.mean(heater_residual(X, d['q1'], d['q2'], P, d['TaK']) ** 2)
            if not sensor_exact: Lr = Lr + torch.mean(sensor_residual(X, P) ** 2)
            L0 = L0 + torch.mean(X[0, :2 if sensor_exact else 4] ** 2)
        n = len(data); L = (Ld + w_r * Lr + w_0 * L0) / n
        return (L, Ld / n, Lr / n, L0 / n) if parts else L

    log, t0 = [], time.time()
    def record(stage, it):
        with torch.no_grad(): L, Ld, Lr, L0 = loss_fn(True)
        rec = dict(stage=stage, it=it, L=float(L), L_data=float(Ld), L_res=float(Lr), L_0=float(L0),
                   t_s=time.time() - t0, **P.as_dict()); log.append(rec)
        if verbose: print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items()})

    # 0) arranque de las redes solo con datos (x_H y x_S cerca de la medición)
    opt = torch.optim.Adam(nets.parameters(), lr=lr)
    for it in range(max(n_adam // 4, 1)):
        opt.zero_grad(); Ls = 0.0
        for d in data:
            X = states(d); Ls = Ls + torch.mean((X[:, 2:] - d['y'])**2) + torch.mean((X[:, :2] - d['y'])**2)
        Ls.backward(); opt.step()
    # 1) Adam conjunto (w, theta)
    opt = torch.optim.Adam([{'params': nets.parameters(), 'lr': lr}, {'params': P.parameters(), 'lr': 1e-2}])
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.1 ** (1.0 / max(n_adam, 1)))
    for it in range(n_adam):
        opt.zero_grad(); L = loss_fn(); L.backward(); opt.step(); sched.step()
        if it % 500 == 0: record('adam', it)
    record('adam', n_adam)
    # 2) L-BFGS
    if n_lbfgs > 0:
        opt = torch.optim.LBFGS(list(nets.parameters()) + list(P.parameters()), lr=1.0, max_iter=n_lbfgs,
                                history_size=50, line_search_fn='strong_wolfe', tolerance_grad=1e-10, tolerance_change=1e-13)
        def closure():
            opt.zero_grad(); L = loss_fn(); L.backward(); return L
        opt.step(closure); record('lbfgs', n_lbfgs)
    return P.as_dict(), log, time.time() - t0, (nets, data, states)

## 3. Modelos de referencia

- **Físico entero / fraccionario (OE-LS):** el mismo modelo 2×2, ajustado por mínimos cuadrados de error de salida
  (simulación completa, como el "Físico" de NB01). El fraccionario por OE-LS es la alternativa directa a la fPINN con
  el mismo modelo: si da lo mismo, el revisor preguntará qué aporta la red (ver notas al final).
- **FOPDT, SOPDT, FFOPDT** sobre G11 (Q1→T1) con los mismos minutos del experimento A. El SOPDT es el modelo de B4 y el
  FFOPDT el de B1; se exportan para NB03.
- **IC 95 %** de ϑ̂ (también para la fPINN): linealización del error de salida del modelo identificado sobre los datos de entrenamiento.

In [ ]:
def oe_residuals(th, exps, L=None):
    out = []
    for e in exps:
        Ta = float(np.mean(np.r_[e['T1'][:5], e['T2'][:5]]))
        T1, T2, _ = sim_frac(e['Q1'], e['Q2'], th, Ta, L=L)
        out += [T1 - e['T1'], T2 - e['T2']]
    return np.concatenate(out)

def ci_linear(th, exps, names, L=None, rel=1e-4):
    """Error estándar de los parámetros 'names' por el jacobiano del error de salida (diferencias finitas)."""
    r0 = oe_residuals(th, exps, L); J = []
    for k in names:
        d = dict(th); dh = rel * max(abs(th[k]), 1e-3); d[k] = th[k] + dh
        J.append((oe_residuals(d, exps, L) - r0) / dh)
    J = np.array(J).T; s2 = r0 @ r0 / max(len(r0) - len(names), 1)
    return dict(zip(names, np.sqrt(np.diag(s2 * np.linalg.pinv(J.T @ J)))))

def fit_phys_oe(exps, frac, L=None):
    names = TH_NAMES if frac else TH_NAMES[:4]
    lb = [0.5, 1e-3, 1e-3, 1.0] + ([0.3, 0.3] if frac else []); ub = [50, 0.05, 0.05, 200] + ([1.2, 1.2] if frac else [])
    x0 = np.clip([TH0[k] for k in names], np.array(lb) * 1.001, np.array(ub) * 0.999)
    mk = lambda x: {**dict(zip(names, x)), **({} if frac else dict(alpha=1.0, beta=1.0))}
    sol = least_squares(lambda x: oe_residuals(mk(x), exps, L), x0, bounds=(lb, ub), x_scale='jac', max_nfev=200)
    th = mk(sol.x); return th, ci_linear(th, exps, names, L)

def delayed(u, theta):
    d = theta / TS; n = int(np.floor(d)); f = d - n; N = len(u); u0 = np.zeros(N); u1 = np.zeros(N)
    if n < N: u0[n:] = u[:N - n]
    if n + 1 < N: u1[n + 1:] = u[:N - n - 1]
    return (1 - f) * u0 + f * u1

def sim_fopdt(p, u):
    K, tau, th = p; ud = delayed(u, th); a = np.exp(-TS / tau); y = np.zeros(len(u))
    for k in range(1, len(u)): y[k] = a * y[k - 1] + K * (1 - a) * ud[k - 1]
    return y

def sim_sopdt(p, u):
    K, t1, t2, th = p; ud = delayed(u, th); a1, a2 = np.exp(-TS / t1), np.exp(-TS / t2)
    x = np.zeros(len(u)); y = np.zeros(len(u))
    for k in range(1, len(u)):
        x[k] = a1 * x[k - 1] + K * (1 - a1) * ud[k - 1]; y[k] = a2 * y[k - 1] + (1 - a2) * x[k - 1]
    return y

def sim_ffopdt(p, u):
    K, tau_a, al, th = p; ud = delayed(u, th); N = len(u)
    g = gl_w(al, N); c = tau_a * TS ** (-al); y = np.zeros(N)
    for k in range(1, N): y[k] = (K * ud[k - 1] - c * (g[1:k + 1] @ y[k - 1::-1])) / (c + 1.0)
    return y

SISO = {'FOPDT':  (sim_fopdt,  ['K', 'tau', 'theta'],          [1e-3, 1, 0],         [5, 3000, 300],        [0.8, 150, 20]),
        'SOPDT':  (sim_sopdt,  ['K', 'tau1', 'tau2', 'theta'], [1e-3, 1, 0.5, 0],    [5, 3000, 1000, 300],  [0.8, 140, 20, 2]),
        'FFOPDT': (sim_ffopdt, ['K', 'tau_a', 'alpha', 'theta'], [1e-3, 0.5, 0.3, 0], [5, 1e5, 1.9, 300],   [0.8, 150, 1.0, 20])}

def fit_siso(name, u, y, n_starts=SISO_STARTS, seed=0):
    fn, names, lb, ub, x0 = SISO[name]; rng = np.random.default_rng(seed); best = None
    for s in range(n_starts):
        p0 = np.array(x0, float) * (1 if s == 0 else rng.uniform(0.6, 1.5, len(x0)))
        if name == 'FFOPDT' and s > 0: p0[2] = rng.uniform(0.6, 1.2); p0[1] = 150 ** p0[2]
        p0 = np.clip(p0, np.array(lb) + 1e-6, np.array(ub) - 1e-6)
        r = least_squares(lambda p: fn(p, u) - y, p0, bounds=(lb, ub), x_scale='jac', max_nfev=300)
        if best is None or r.cost < best.cost: best = r
    return dict(zip(names, best.x))

## 4. Una corrida: entrenar, validar y aplicar el criterio de H1

`run_config(caso, σn, D, semilla)` devuelve una fila por modelo. El RMSE de validación de la fPINN y de la PINN entera se
calcula **simulando el modelo identificado** (sustituto, paso 1 s, memoria `L_SUR`), no con la red, porque eso es lo que usa NB03.

In [ ]:
def val_rmse_2x2(th, case):
    out = {}
    for nm in ['V1', 'V2']:
        e = PLANT[case][nm]; T1, T2, _ = sim_frac(e['Q1'], e['Q2'], th, PHYS['Ta'], L=L_SUR)
        out[f'RMSE_{nm}_T1'] = float(np.sqrt(np.mean((T1 - e['T1'])**2)))
        out[f'RMSE_{nm}_T2'] = float(np.sqrt(np.mean((T2 - e['T2'])**2)))
    return out

def val_rmse_siso(name, par, case):
    e = PLANT[case]['V1']; y = e['T1'] - PHYS['Ta']; fn, names, *_ = SISO[name]
    return {'RMSE_V1_T1': float(np.sqrt(np.mean((fn([par[k] for k in names], e['Q1']) - y)**2)))}

def order_flag(v, se):
    """Orden fraccionario significativo: |v-1| >= tol, IC 95 % excluye 1 y v < 1 (un orden > 1 no cuenta)."""
    return bool(abs(v - 1) >= ORDER_TOL and abs(v - 1) > 1.96 * se and v < 1)

def run_config(case, sigma, D, seed, do_int_pinn, do_siso):
    exps = noisy_data(case, sigma, seed, D)
    base = dict(alpha_true=case[0], beta_true=case[1], noise=sigma, D_min=D, seed=seed)
    rows, sur = [], {}
    # físico entero y fraccionario por OE-LS
    t0 = time.time(); th_i, se_i = fit_phys_oe(exps, frac=False)
    rows.append(dict(base, model='Phys-int (OE-LS)', time_s=time.time() - t0, **th_i, **{f'se_{k}': v for k, v in se_i.items()},
                     **val_rmse_2x2(th_i, case)))
    t0 = time.time(); th_f, se_f = fit_phys_oe(exps, frac=True)
    rows.append(dict(base, model='Phys-frac (OE-LS)', time_s=time.time() - t0, **th_f, **{f'se_{k}': v for k, v in se_f.items()},
                     **val_rmse_2x2(th_f, case)))
    # fPINN
    th_p, log, tt, _ = train_pinn(exps, seed=seed, **PINN)
    se_p = ci_linear(th_p, exps, TH_NAMES, L_SUR)
    rows.append(dict(base, model='fPINN', time_s=tt, **th_p, **{f'se_{k}': v for k, v in se_p.items()},
                     L_data=log[-1]['L_data'], L_res=log[-1]['L_res'], **val_rmse_2x2(th_p, case)))
    sur['fPINN'] = th_p
    # PINN entera (B3)
    if do_int_pinn:
        th_ip, log_i, tt, _ = train_pinn(exps, seed=seed, learn_orders=False, **PINN)
        rows.append(dict(base, model='Integer PINN', time_s=tt, **th_ip, L_data=log_i[-1]['L_data'],
                         L_res=log_i[-1]['L_res'], **val_rmse_2x2(th_ip, case)))
        sur['IntegerPINN'] = th_ip
    # modelos SISO sobre G11 (primeros D min del experimento A)
    if do_siso:
        eA = exps[0]; u = eA['Q1'].astype(float); y = eA['T1'] - eA['T1'][:5].mean()
        for name in SISO:
            t0 = time.time(); par = fit_siso(name, u, y, seed=seed)
            rows.append(dict(base, model=name, time_s=time.time() - t0, **par, **val_rmse_siso(name, par, case)))
            sur[name] = par
    # criterio de H1 (fPINN frente al físico entero, RMSE de V1-T1 y V2)
    r_int = rows[0]; r_p = rows[2]
    gain = 100 * (np.mean([r_int['RMSE_V1_T1'], r_int['RMSE_V2_T1'], r_int['RMSE_V2_T2']]) -
                  np.mean([r_p['RMSE_V1_T1'], r_p['RMSE_V2_T1'], r_p['RMSE_V2_T2']])) / \
           np.mean([r_int['RMSE_V1_T1'], r_int['RMSE_V2_T1'], r_int['RMSE_V2_T2']])
    fa, fb = order_flag(th_p['alpha'], se_p['alpha']), order_flag(th_p['beta'], se_p['beta'])
    r_p.update(gain_vs_int_pct=gain, frac_alpha=fa and gain > GAIN_TOL, frac_beta=fb and gain > GAIN_TOL,
               verdict=('EVIDENCIA' if (fa or fb) and gain > GAIN_TOL else ('débil' if (fa or fb or gain > GAIN_TOL) else 'sin evidencia')))
    return rows, sur

## 5. Barrido (se guarda y retoma)

In [ ]:
def config_list():
    """Primero la configuración principal (tabla tab:sim_id y sustitutos para NB03), luego el resto del barrido."""
    allc = list(itertools.product(CASES, NOISES, DURATIONS_MIN, SEEDS))
    is_main = lambda c: c[1] == MAIN['noise'] and c[2] == MAIN['D']
    for case, sigma, D, seed in sorted(allc, key=lambda c: not is_main(c)):
        main = is_main((case, sigma, D, seed))
        yield case, sigma, D, seed, (RUN_INT_PINN_ALL or main), (RUN_SISO_ALL or main)

done = set()
if os.path.exists(RES_CSV):
    old = pd.read_csv(RES_CSV)
    done = set(map(tuple, old[old.model == 'fPINN'][['alpha_true', 'beta_true', 'noise', 'D_min', 'seed']].values.tolist()))
todo = [c for c in config_list() if (c[0][0], c[0][1], c[1], c[2], c[3]) not in done]
print(f'{len(todo)} corridas pendientes de {len(list(config_list()))}')

t_start = time.time()
for i, (case, sigma, D, seed, do_ip, do_siso) in enumerate(todo):
    t0 = time.time()
    rows, sur = run_config(case, sigma, D, seed, do_ip, do_siso)
    df = pd.DataFrame(rows)
    df.to_csv(RES_CSV, mode='a', header=not os.path.exists(RES_CSV), index=False)
    if do_ip or do_siso:     # sustitutos de la configuración principal -> NB03
        tag = f"a{case[0]}_b{case[1]}_s{seed}"
        with open(os.path.join(OUT_DIR, 'surrogates', f'{tag}.json'), 'w') as f:
            json.dump(dict(case=case, noise=sigma, D_min=D, seed=seed, phys=PHYS, h=TS, L=L_SUR, models=sur), f, indent=1)
    p = df[df.model == 'fPINN'].iloc[0]
    print(f'[{i + 1}/{len(todo)}] caso {case} σ={sigma} D={D} s={seed}: α̂={p.alpha:.3f} β̂={p.beta:.3f} U={p.U:.2f} '
          f'τs={p.tau_s:.1f}  RMSE_V1={p.RMSE_V1_T1:.3f}  {p.verdict}  ({time.time() - t0:.0f} s; total {(time.time() - t_start) / 60:.1f} min)')
res = pd.read_csv(RES_CSV)

## 6. Tabla `tab:sim_id` (σn = 0,1 °C y 30 min en `full`; media ± desv. estándar sobre semillas)

Se guarda como CSV y como filas LaTeX listas para pegar en el manuscrito.

In [ ]:
m = res[(res.noise == MAIN['noise']) & (res.D_min == MAIN['D'])]
pm = lambda s, d=3: f'{s.mean():.{d}f} \\pm {s.std(ddof=1):.{d}f}' if len(s) > 1 else f'{s.mean():.{d}f}'
tab, tex = [], []
for case in CASES:
    mc = m[(m.alpha_true == case[0]) & (m.beta_true == case[1])]
    g = lambda name: mc[mc.model == name]
    row = dict(case=str(case), alpha_hat=pm(g('fPINN').alpha), beta_hat=pm(g('fPINN').beta),
               RMSE_fPINN=pm(g('fPINN').RMSE_V1_T1), RMSE_int_phys=pm(g('Phys-int (OE-LS)').RMSE_V1_T1),
               RMSE_FFOPDT=pm(g('FFOPDT').RMSE_V1_T1) if len(g('FFOPDT')) else 'n/a',
               RMSE_frac_OE=pm(g('Phys-frac (OE-LS)').RMSE_V1_T1),
               RMSE_int_PINN=pm(g('Integer PINN').RMSE_V1_T1) if len(g('Integer PINN')) else 'n/a',
               verdicts='/'.join(g('fPINN').verdict.astype(str)))
    tab.append(row)
    tex.append(f"$({case[0]:g}, {case[1]:g})$ & ${row['alpha_hat']}$ & ${row['beta_hat']}$ & ${row['RMSE_fPINN']}$ & "
               f"${row['RMSE_int_phys']}$ & ${row['RMSE_FFOPDT']}$ \\\\")
tab = pd.DataFrame(tab); tab.to_csv(os.path.join(OUT_DIR, 'tab_sim_id.csv'), index=False)
open(os.path.join(OUT_DIR, 'tab_sim_id_rows.tex'), 'w').write('\n'.join(tex) + '\n')
print(tab.to_string(index=False)); print(); print('\n'.join(tex))

# todos los modelos (para el texto: FOPDT, SOPDT, físico fraccionario OE-LS, PINN entera, tiempos)
summ = m.groupby(['alpha_true', 'beta_true', 'model'])[['RMSE_V1_T1', 'RMSE_V2_T1', 'RMSE_V2_T2', 'time_s']].agg(['mean', 'std'])
summ.to_csv(os.path.join(OUT_DIR, 'resumen_modelos_main.csv')); print(summ.round(3))

## 7. Figura: error de orden y RMSE contra la duración de los datos (una curva por nivel de ruido)

In [ ]:
f = res[res.model == 'fPINN'].copy()
f['err_beta'] = (f.beta - f.beta_true).abs(); f['err_alpha'] = (f.alpha - f.alpha_true).abs()
fig, ax = plt.subplots(len(CASES), 2, figsize=(7, 2.2 * len(CASES)), squeeze=False, sharex=True)
for i, case in enumerate(CASES):
    fc = f[(f.alpha_true == case[0]) & (f.beta_true == case[1])]
    for sigma in NOISES:
        g = fc[fc.noise == sigma].groupby('D_min')
        for j, col in enumerate(['err_beta', 'RMSE_V1_T1']):
            mu, sd = g[col].mean(), g[col].std().fillna(0)
            ax[i, j].errorbar(mu.index, mu.values, yerr=sd.values, marker='o', ms=3, capsize=2, label=f'σn = {sigma} °C')
    ax[i, 0].set_ylabel(f'({case[0]:g}, {case[1]:g})\n|β̂ − β*|'); ax[i, 1].set_ylabel('RMSE V1 [°C]')
for j in range(2): ax[-1, j].set_xlabel('training data per experiment [min]')
ax[0, 1].legend(fontsize=7); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_sim_id_data.pdf')); plt.savefig(os.path.join(OUT_DIR, 'fig_sim_id_data.png'), dpi=150); plt.show()

## 8. Veredicto H1 por caso (configuración principal) y sustitutos para NB03

Resultado esperado: (1, 1) → "sin evidencia" (prueba de control: el método no inventa fraccionalidad);
(1, 0,7) y (0,9, 0,7) → "EVIDENCIA", con α̂ y β̂ cerca de los valores reales.
Los JSON de `NB02_out/surrogates/` contienen, por caso y semilla, ϑ̂ de la fPINN (sustituto de P, B2 y A1–A2),
de la PINN entera (B3), del SOPDT (B4) y del FFOPDT (B1).

In [ ]:
fm = f[(f.noise == MAIN['noise']) & (f.D_min == MAIN['D'])]
ver = fm.groupby(['alpha_true', 'beta_true']).agg(alpha_hat=('alpha', 'mean'), beta_hat=('beta', 'mean'),
                                                  se_beta=('se_beta', 'mean'), gain_pct=('gain_vs_int_pct', 'mean'),
                                                  evidencia=('verdict', lambda v: f"{(v == 'EVIDENCIA').sum()}/{len(v)}"))
ver.to_csv(os.path.join(OUT_DIR, 'veredicto_H1_sim.csv')); print(ver.round(3))
print('Sustitutos exportados:', sorted(os.listdir(os.path.join(OUT_DIR, 'surrogates'))))

## 9. Notas para la discusión

- **Memoria del sustituto.** La celda de la sección 1 compara el sustituto (paso 1 s) con la planta: con memoria
  completa el error es de centésimas de grado, pero truncar la memoria a 900 s da ≈ 0,7 °C en el caso (0,9, 0,7) porque
  el calentador fraccionario tiene memoria larga. Por eso `L_SUR = None` (memoria completa); para NB03 conviene
  L ≥ 3000 s o memoria completa. El "short-memory principle" no es inocuo aquí.
- **fPINN frente a OE-LS con el mismo modelo.** El ajuste por error de salida del mismo modelo fraccionario es rápido y,
  con datos limpios de simulación, preciso. El argumento a favor de la fPINN debe apoyarse en lo que esta tabla mide
  (robustez al arranque, al ruido y a pocos datos, y estados latentes x_H) y en el costo; si no aparece ventaja, hay que
  decirlo en el artículo y presentar la fPINN como un medio para obtener el sustituto diferenciable.
- Un orden > 1 se interpreta como orden entero faltante (igual que NB01) y no cuenta como fraccionalidad.

## 10. Empaquetar resultados

Crea `NB02_sim_resultados_<fecha>.zip` con todo `NB02_out/` (sin la subcarpeta `kit/` del NB02 del kit). Ese .zip es el que se sube al proyecto.

In [ ]:
import zipfile, datetime
zname = f"NB02_sim_resultados_{datetime.date.today().strftime('%Y%m%d')}{'_quick' if MODE == 'quick' else ''}.zip"
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT_DIR):
        dirs[:] = [d for d in dirs if not (root == OUT_DIR and d in ('kit', 'NB01_desde_zip'))]
        for fn in files: z.write(os.path.join(root, fn))
print('Guardado:', os.path.abspath(zname), f'({os.path.getsize(zname) / 1e6:.1f} MB)')